# Como os clientes se agrupam ao longo do tempo: StoCFL e CFL

Este notebook desenha, com Mermaid, a evolução dos clusters em cada algoritmo. Os dois exemplos são **ilustrativos**, e não uma execução real: servem para mostrar o sentido do movimento, que é oposto nos dois.

| | StoCFL | CFL |
|---|---|---|
| Ponto de partida | cada cliente em seu próprio cluster (K = N) | um único cluster com todos (K = 1) |
| Operação | só **união** | só **divisão** |
| Quando age | a cada rodada, entre os clusters dos clientes sorteados | quando as normas das atualizações indicam clientes incongruentes |
| Critério | cosseno entre os Ψ dos clusters maior que τ | `mean_norm < ε₁`, `max_norm > ε₂` e `γ_max < √((1 − α_cross_max) / 2)` |
| Desfazer | não há | não há (clusters não se fundem) |

Referências: [guia técnico](guia.md), seções 3.1 e 3.3 (CFL) e 4.15.2 (StoCFL); diagramas de atividades de [CFL](../projetos/clustered-federated-learning/diagrama-atividades-cfl.md) e de [StoCFL](../projetos/StoCFL/diagrama-atividades-stocfl.md).

Os diagramas são desenhados com o `mermaid.js` baixado de um CDN, então a visualização precisa de internet. Execute a célula abaixo antes das outras.

In [4]:
import html
from IPython.display import HTML, display

MERMAID_JS = "https://cdn.jsdelivr.net/npm/mermaid@10.9.1/dist/mermaid.min.js"

def mermaid(codigo, altura=600):
    """Desenha um diagrama Mermaid num iframe (precisa de internet para baixar o mermaid.js).

    Depois de desenhar, o iframe ajusta a própria altura ao tamanho do diagrama.
    `altura` só vale até o desenho terminar (ou se o ajuste automático for bloqueado).
    """
    pagina = (
        "<!doctype html><meta charset='utf-8'>"
        "<body style='margin:0;background:#fff;font-family:sans-serif'>"
        f"<pre class='mermaid'>{html.escape(codigo)}</pre>"
        f"<script src='{MERMAID_JS}'></script>"
        "<script>"
        "mermaid.initialize({startOnLoad:false, securityLevel:'loose',"
        " flowchart:{useMaxWidth:true, htmlLabels:true, nodeSpacing:30, rankSpacing:45},"
        " themeVariables:{fontSize:'15px'}});"
        "mermaid.run().then(function(){try{"
        "window.frameElement.style.height=(document.body.scrollHeight+8)+'px';"
        "}catch(e){}});"
        "</script>"
    )
    display(HTML(
        f'<iframe srcdoc="{html.escape(pagina, quote=True)}" '
        f'style="width:100%;height:{altura}px;border:0;background:#fff"></iframe>'
    ))

## StoCFL: clusters se unindo

Exemplo: 8 clientes com MNIST rotacionado em quatro rotações (0°, 90°, 180°, 270°), dois clientes por rotação. Cada linha é uma rodada e cada caixa é um cluster.

- A **cor** é a rotação real dos dados (azul 0°, laranja 90°, verde 180°, rosa 270°). O algoritmo não conhece essa cor, ele só vê o cosseno entre os Ψ.
- **Borda grossa** marca o cluster que acabou de ser formado por uma união.
- **Tracejado** são clusters sem nenhum membro sorteado na rodada, que ficam como estavam.

In [5]:
STOCFL = """
flowchart TB
    classDef r0 fill:#e1effd,stroke:#2e6fb7,color:#16232a
    classDef r90 fill:#fff1d6,stroke:#b9770e,color:#16232a
    classDef r180 fill:#d6efeb,stroke:#09766e,color:#16232a
    classDef r270 fill:#f6dde8,stroke:#a23b72,color:#16232a
    classDef idle stroke-dasharray:4 3,fill:#ffffff,stroke:#8a9aa5,color:#5b7083
    classDef uniao stroke:#16232a,stroke-width:4px

    subgraph T0["Início · K = 8 · cada cliente num cluster próprio"]
        direction LR
        a0["{0}"]:::r0
        a1["{1}"]:::r0
        a2["{2}"]:::r90
        a3["{3}"]:::r90
        a4["{4}"]:::r180
        a5["{5}"]:::r180
        a6["{6}"]:::r270
        a7["{7}"]:::r270
    end

    subgraph T1["Rodada 1 · sorteados 0, 2, 3, 5 · K = 7"]
        direction LR
        b0["{0}"]:::r0
        b1["{1}"]:::idle
        b23["{2, 3}<br/>união"]:::r90
        b4["{4}"]:::idle
        b5["{5}"]:::r180
        b6["{6}"]:::idle
        b7["{7}"]:::idle
    end

    subgraph T2["Rodada 2 · sorteados 1, 3, 4, 7 · K = 7 · nenhuma união"]
        direction LR
        c0["{0}"]:::idle
        c1["{1}"]:::r0
        c23["{2, 3}"]:::r90
        c4["{4}"]:::r180
        c5["{5}"]:::idle
        c6["{6}"]:::idle
        c7["{7}"]:::r270
    end

    subgraph T3["Rodada 3 · sorteados 0, 1, 4, 5 · K = 5"]
        direction LR
        d01["{0, 1}<br/>união"]:::r0
        d23["{2, 3}"]:::idle
        d45["{4, 5}<br/>união"]:::r180
        d6["{6}"]:::idle
        d7["{7}"]:::idle
    end

    subgraph T4["Rodada 4 · sorteados 2, 6, 7 · K = 4"]
        direction LR
        e01["{0, 1}"]:::idle
        e23["{2, 3}"]:::r90
        e45["{4, 5}"]:::idle
        e67["{6, 7}<br/>união"]:::r270
    end

    T0 --> T1 --> T2 --> T3 --> T4

    class b23,d01,d45,e67 uniao
"""

mermaid(STOCFL, 900)

Como ler o exemplo:

- **Rodada 2** não funde nada: o cliente 0, par do 1, não foi sorteado, e o cosseno do 1 com os outros sorteados não passa de τ.
- **K só diminui** (de 8 a 4). Nenhum cluster se divide, então uma união errada não é desfeita.
- **Ψ é calculado uma vez**, na primeira vez que o cliente é sorteado: o gradiente normalizado da perda no modelo âncora (ω₀). O centro do cluster é a média dos Ψ de **todos** os seus membros, inclusive dos que não foram sorteados na rodada (Eq. 2 do artigo; correção 8 de 4.12 do guia).
- **Dentro de uma rodada**, os pares de clusters sorteados são percorridos do maior cosseno ao menor, e o primeiro par abaixo de τ encerra as uniões. Os cossenos não são recalculados depois de cada união. O cluster resultante recebe o menor id.

## CFL: um cluster que se divide

Exemplo: o cenário do notebook do CFL, com 10 clientes em EMNIST não-IID. Os clientes 0 a 4 têm as imagens rotacionadas em 180° e os clientes 5 a 9 não, então há dois grupos reais. O número da rodada `r*` é arbitrário.

- Cinza é um cluster que mistura os dois grupos. Depois da divisão, verde é o grupo rotacionado e azul é o grupo sem rotação.

In [6]:
CFL = """
flowchart TB
    classDef mix fill:#eef2f7,stroke:#5b7083,color:#16232a
    classDef g180 fill:#d6efeb,stroke:#09766e,color:#16232a
    classDef g0 fill:#e1effd,stroke:#2e6fb7,color:#16232a
    classDef split fill:#fff1d6,stroke:#b9770e,stroke-width:2px,color:#16232a
    classDef fim fill:#f6dde8,stroke:#a23b72,color:#16232a

    subgraph T0["Início · K = 1 · FedAvg comum"]
        direction LR
        a["{0, 1, 2, 3, 4, 5, 6, 7, 8, 9}<br/>clientes 0 a 4 rotacionados 180° · clientes 5 a 9 sem rotação"]:::mix
    end

    subgraph T1["Rodadas iniciais · K = 1 · sem divisão"]
        direction LR
        b["{0, ..., 9}<br/>mean_norm ainda alta · só FedAvg"]:::mix
    end

    subgraph T2["Rodada r* · o cluster se divide"]
        direction LR
        c1["1. Critério<br/>mean_norm #lt; ε₁<br/>max_norm #gt; ε₂"]:::split
        c2["2. Bipartição<br/>Algoritmo 1 sobre<br/>S = cos(ΔW)"]:::split
        c3["3. Teste<br/>γ_max #lt; √((1 − α_cross_max) / 2)<br/>aceita e faz cache_model do pai"]:::split
        c1 --> c2 --> c3
    end

    subgraph T3["Depois da divisão · K = 2"]
        direction LR
        d1["{0, 1, 2, 3, 4}<br/>FedAvg próprio"]:::g180
        d2["{5, 6, 7, 8, 9}<br/>FedAvg próprio"]:::g0
    end

    subgraph T4["Fim"]
        direction LR
        e["cache_model dos clusters finais · cada cliente escolhe<br/>o modelo de maior acurácia (base ou de cluster)"]:::fim
    end

    T0 --> T1 --> T2 --> T3 --> T4
"""

mermaid(CFL, 900)

Como ler o exemplo:

- **Rodadas iniciais:** com um só cluster, o treino é FedAvg comum. O servidor mede `mean_norm` (norma da média das atualizações) e `max_norm` (maior norma individual) a cada rodada.
- **Rodada r\*:** o modelo chegou perto de um ponto estacionário do objetivo conjunto (`mean_norm` pequena), mas cada cliente ainda está longe do seu próprio ótimo (`max_norm` grande). Só então o servidor faz a bipartição por similaridade de cosseno.
- **A divisão pode ser rejeitada.** Se `γ_max` não for menor que `√((1 − α_cross_max) / 2)`, o cluster fica como está e o teste se repete na rodada seguinte.
- **Divisões seguintes:** cada cluster novo pode se dividir de novo pelo mesmo critério (desde que tenha mais de um cliente). O cenário do notebook, com a seed 42, faz uma só divisão.
- **Fim:** `cache_model` guarda o modelo de cada cluster final, e cada cliente escolhe, entre os modelos guardados (o do cluster pai inclusive), o de maior acurácia na sua validação.